# Week 9a.1: Building an Agent Harness

So far, every agent we've built runs the same loop from Week 4: the model calls tools, reads the results, and calls more tools until it can answer. That loop works well when a task takes a handful of steps. 

Long-running tasks are harder for it. Every tool result stays in the message list, so the context keeps growing, and the agent has no record of what it planned to do other than the conversation itself. In practice, a single agent on a long task tends to stop early and answer only part of it.

An **agent harness** is the code we put around the model loop so that it can work on long tasks. Coding agents such as Claude Code and Codex are harnesses: the model is one part, and the rest is tools, files, instructions, and rules about what the agent may do.

Today we'll build three pieces that most harnesses share on top of the research agent from Week 7:

| Piece | What it gives the agent |
|---|---|
| A to-do list | a plan that it writes down and updates as it works |
| Files | a place to keep notes outside the message list |
| A `task` tool | a way to hand a piece of work to a new subagent with a fresh context |

In the next notebook, we'll use `deepagents`, LangChain's library that packages these pieces.

In [ ]:
from dotenv import load_dotenv
import os

load_dotenv()
assert os.getenv("GOOGLE_API_KEY"), "No GOOGLE_API_KEY found."
print("API key loaded")

## 0. Model Setup

Execute one of the following to define the model.

In [ ]:
from langchain_ollama import ChatOllama

model = ChatOllama(model="qwen3.5:9b", reasoning=False)

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="gpt-4.1-mini")

In [ ]:
from langchain.chat_models import init_chat_model

model = init_chat_model(model="claude-haiku-5-5")

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

model = ChatGoogleGenerativeAI(model="gemini-3.5-flash-lite")

In [ ]:
from langchain_nvidia_ai_endpoints import ChatNVIDIA
model = ChatNVIDIA(model="nvidia/nemotron-3.5-super-vl-120b-a12b")
#model = ChatNVIDIA(model="nvidia/nemotron-3.5-lightning-30b-a3b")

NOTE:

The harness at the end of this notebook runs a lead agent and three subagents. One run makes about 20 model calls in under a minute, and the Gemini free tier allows 15 calls per minute. If you see a 429 error, wait a minute and run the cell again, or use one of the other models.

## 1. A long task for one agent

We'll reuse the paper tools from Week 7, `search_papers` and `get_paper`, over the cached `papers.json`. To make the task long, we'll combine three of the five sample questions into one request: a short literature review that has to answer all three.

The gold list for the combined task is the union of the three questions' gold lists.

In [ ]:
import time
from langchain.messages import HumanMessage, SystemMessage, ToolMessage

import research_tools as rt
import research_team as team

qs = list(rt.questions())
parts = [qs[3], qs[2], qs[4]]
QUESTION = "Write a short literature review for a newcomer on three questions. \n " + " ".join(parts)
GOLD = sorted(set(g for q in parts for g in rt.questions()[q]["gold"]))

print(QUESTION)
print(f"\n{len(GOLD)} gold papers")

Let's start with the single agent from W07a.2. It is the Week 4 loop with the two paper tools, available as `team.build_single`.

In [ ]:
single = team.build_single(model)
one = team.run_single(single, QUESTION)

print(f"{len(one['calls'])} tool calls, {one['tokens']:,} tokens, {one['seconds']:.0f} s")
print(f"coverage: {rt.score_coverage(one['report'], GOLD):.2f}\n")
for name, arg in one["calls"]:
    print(f"  {name:14} {arg}")

Now let's look at the size of the context each time the model was called. `input_tokens` is the number of tokens the model read on that call.

In [ ]:
def input_tokens(messages):
    return [m.usage_metadata["input_tokens"] for m in messages if getattr(m, "usage_metadata", None)]

input_tokens(one["messages"])

Two things to notice here.

- **The context only grows.** Every search result and every abstract the agent read is still in the message list on the next call, whether it still matters or not.
- **Count the tool calls against the three questions.** A review that answers three questions needs at least three searches and a few papers for each. Check the calls above to see whether the agent did that, or whether it wrote the report after covering only part of the request.

The model was able to do each part. What it lacked was a structure for a task with several parts: something that keeps track of what is left to do, and a way to work on one part without carrying the other parts' search results along.

## 2. The harness state

The harness keeps three things in the graph state alongside the messages:

- `todos`: the agent's to-do list
- `files`: a dictionary from a path such as `/notes/memory.md` to the text saved there
- `subagent_tokens`: the tokens the subagents used, so we can count the cost of the whole run

The files live in the graph state, not on your disk. They are saved by the checkpointer like everything else in the state, and they disappear when the thread does.

`files` needs a reducer. Several subagents will run at the same time and each will save a file, so several updates to `files` arrive in the same step. Without a reducer, LangGraph raises `InvalidUpdateError`, as we saw with parallel branches in Week 7. `merge_files` combines the dictionaries, so every file is kept.

In [ ]:
import operator
from typing import Annotated, Literal
from langgraph.graph import MessagesState


def merge_files(old: dict, new: dict) -> dict:
    return {**old, **new}


class HarnessState(MessagesState):
    # TODO

## 3. A to-do list

The to-do list is a tool, `write_todos`. Its argument is the whole list, with a status for each item: `pending`, `in_progress`, or `completed`. 

The agent updates the list by calling the tool again with the new list.

Sending the whole list each time is simpler for the model than separate tools to add, remove, and check off items, and every call leaves the full current plan in the message list.

The tool returns a `Command`, as the handoff tools in W08a did. The `Command` writes the list into `todos` in the state, and it writes the tool's own `ToolMessage`, because a tool that returns a `Command` is responsible for its `ToolMessage`.

In [ ]:
from pydantic import BaseModel
from langchain.tools import tool, ToolRuntime
from langgraph.types import Command


class Todo(BaseModel):
    content: str
    status: Literal["pending", "in_progress", "completed"]



In [ ]:
todo_item = {"content": "Find papers on tool use", "status": "pending"}


In [ ]:
t = Todo(**todo_item)

print(type(t))
print(t)                 # content='Find papers on tool use' status='pending'
print(t.status)          # pending      (attribute access works)

In [ ]:
# Todo object -> dictionary
t_dict = t.model_dump()
print(t_dict)    # {'content': 'Find papers on tool use', 'status': 'pending'}

In [ ]:
print(f"[{t_dict['status']}] {t_dict['content']}")

In [ ]:
@tool
def write_todos(todos: list[Todo], runtime: ToolRuntime) -> Command:
    """Write your to-do list for the task. Send the whole list every time, with each item's status:
    pending, in_progress, or completed."""
    items = [t.model_dump() for t in todos]
    shown = "\n".join(f"[{t['status']}] {t['content']}" for t in items)
    return Command(update={"todos": items,
                           "messages": [ToolMessage(f"To-do list updated:\n{shown}",
                                                    tool_call_id=runtime.tool_call_id)]})

### Tool arguments are structured output

`write_todos` takes a list of `Todo` objects, not a string, and we never wrote code to parse the model's output into that list. 

So far we've achieved the same kind of result with `with_structured_output`: 
- we gave the model a Pydantic class, and its reply came back as an object of that class. 

A tool's signature does the same job for the tool's arguments.

When we decorate a function with `@tool`, LangChain reads its type hints and turns them into a JSON schema, which is sent to the model along with the tool's name and docstring. 
- For `write_todos`, the schema says that `todos` is a list of objects, each with a string `content` and a `status` that must be one of the three allowed values. 
- When the model calls the tool, it writes arguments that follow the schema. LangChain checks them against `Todo` and converts each item into a `Todo` object before our function runs.

The difference from `with_structured_output` is when the schema applies. 
- With structured output, the model's whole reply has to match the class. 
- A tool's schema applies only when the model chooses to call that tool, so the model can still call other tools or answer in text. 
- This is a common way to get structured data from a model in the middle of an agent loop. The data arrives as the tool's arguments, and the body of the tool decides what to do with it.

Here is the schema the model receives for `write_todos`. `runtime` does not appear in it, because `ToolNode` supplies it.


In [ ]:
write_todos.tool_call_schema.model_json_schema()

Before we give `write_todos` to a model, let's call it ourselves and look at what it does to the state.

`call_tool` builds a graph with only a `ToolNode` and sends it one model message that asks for one tool call. This is what happens inside the agent loop when the model calls a tool. We need the small graph because `ToolNode` gets the `runtime` from the graph that runs it.

The output shows two things: 
- the `ToolMessage` that the model would read on its next call, and 
- the list that `write_todos` stored in `todos`.

In [ ]:
from langchain.messages import AIMessage
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode


def call_tool(tool, args, state):
    builder = StateGraph(HarnessState)
    builder.add_node("tools", ToolNode([tool]))
    builder.add_edge(START, "tools")
    request = AIMessage(content="", tool_calls=[{"name": tool.name, "args": args, "id": "call_1"}])
    return builder.compile().invoke({**state, "messages": [request]})


after = call_tool(write_todos,
                  {"todos": [{"content": "Find papers on tool use", "status": "in_progress"},
                             {"content": "Write the report", "status": "pending"}]},
                  {"todos": []})


In [ ]:
print(after["messages"][-1].text)

In [ ]:
print(after["todos"])

## 4. Files

Three file tools: `write_file`, `read_file`, and `list_files`.

- `write_file` changes the state, so it returns a `Command` that updates `files`.
- `read_file` and `list_files` only read the state, so they return a string. 
    - `runtime.state` gives them the current state.

**Why files matter for a team of agents.**
- A subagent can save its full notes in a file and return a short summary to the lead. 
- The lead's context then holds the summaries, and the lead reads a file only when it needs the detail.

In [ ]:
@tool
def write_file(path: str, content: str, runtime: ToolRuntime) -> Command:
    """Save text to a file, replacing it if it exists. Paths look like /notes/memory.md."""
    return Command(update={"files": #TODO,
                           "messages": [ToolMessage(f"Saved {path} ({len(content)} characters).",
                                                    tool_call_id=runtime.tool_call_id)]})


@tool
def read_file(path: str, runtime: ToolRuntime) -> str:
    """Read a file that you or a subagent saved."""
    files = runtime.state.get("files", {})
    return files.get(path, f"No file {path}. Files: {', '.join(files) or 'none'}")


@tool
def list_files(runtime: ToolRuntime) -> str:
    """List the saved files."""
    files = runtime.state.get("files", {})
    return "\n".join(f"{p} ({len(c)} characters)" for p, c in files.items()) or "No files yet."

Now let's try the three file tools:

In [ ]:
files = {"/notes/tools.md": "Toolformer teaches a model to call APIs [2302.04761]."}

after = call_tool(write_file, {"path": "/notes/memory.md", "content": "MemGPT pages memory in and out [2310.08560]."},
                  {"files": files})

print("write_file:", after["messages"][-1].text)
print("files now: ", list(after["files"]))


In [ ]:
after = call_tool(read_file, {"path": "/notes/tools.md"}, after)
print("\nread_file:", after["messages"][-1].text)


In [ ]:
after = call_tool(list_files, {}, after)
print("\nlist_files:")
print(after["messages"][-1].text)

## 5. The loop

Both the lead and the subagents run the loop from Week 4. The only change from `team.build_agent` is the state: `HarnessState` instead of `MessagesState`, so that the tools can read and write `todos` and `files`.

In [ ]:
from langgraph.graph import StateGraph, START
from langgraph.prebuilt import ToolNode, tools_condition


def build_loop(model, prompt, tools):
    model_with_tools = model.bind_tools(tools)
    sys_msg = SystemMessage(content=prompt)

    def assistant(state: HarnessState) -> dict:
        return {"messages": [model_with_tools.invoke([sys_msg] + state["messages"])]}

    builder = StateGraph(HarnessState)
    builder.add_node("assistant", assistant)
    builder.add_node("tools", ToolNode(tools))
    builder.add_edge(START, "assistant")
    builder.add_conditional_edges("assistant", tools_condition)
    builder.add_edge("tools", "assistant")
    return builder.compile()

## 6. The `task` tool

The `task` tool is how the lead creates a subagent. The lead writes a description of the work, and the tool runs a research subagent on that description and returns the subagent's final message.

The subagent is the research loop with the paper tools and `write_file`. Its prompt tells it that the description is all it knows, and asks for its notes in a file and a short summary as its reply.

In [ ]:
RESEARCHER_PROMPT = """You are a research assistant. You get one piece of research to do, and the
instructions are all you know about the larger task.
Search with search_papers, read the two to four papers that matter with get_paper, and save your notes
with write_file at the path the instructions give, with every claim followed by the paper id in brackets
like [2503.13657].
Then reply with a summary of at most 80 words that names the file and the paper ids you cited.
Cite only papers you have read."""

researcher = build_loop(model, RESEARCHER_PROMPT, [rt.search_papers, rt.get_paper, write_file])


In [ ]:
@tool
def task(description: str, runtime: ToolRuntime) -> Command:
    """Give one self-contained piece of research to a new research subagent. It sees only this
    description, so say exactly what to find, which file to save its notes in, and what to report back.
    To run several subagents at once, call task several times in one message."""
    result = researcher.invoke({"messages": [HumanMessage(content=description)],
                                "files": runtime.state.get("files", {})})
    return Command(update={"files": result["files"],
                           "subagent_tokens": team.tokens_of(result["messages"]),
                           "messages": [ToolMessage(result["messages"][-1].text,
                                                    tool_call_id=runtime.tool_call_id)]})

Let's call `task` once with a small piece of work, as the lead would. This starts one research subagent, so it makes a few model calls. The output shows what comes back to the lead: the subagent's reply as a `ToolMessage`, the file it saved, and the tokens it used.

In [ ]:
after = call_tool(task,
                  {"description": "Find two papers on how LLM agents learn to use tools. Save notes to /notes/demo.md."},
                  {"files": {}, "subagent_tokens": 0})

print("reply to the lead:", after["messages"][-1].text)
print("\nfiles back:       ", list(after["files"]))
print("subagent tokens:  ", after["subagent_tokens"])

Let's look at what `task` passes in and what it passes back.

- **In:** a message list with one `HumanMessage`, the description. The subagent does not see the lead's conversation, the question, or the other subagents. This is the same context isolation as the workers in W07a.2, where each worker received a `WorkerState` with one sub-question.
- **In and back:** the files. The subagent starts with the lead's files, so it could read earlier notes, and the files it saves come back to the lead through the `files` reducer.
- **Back:** only the subagent's last message. Its searches and abstracts stay in its own message list, which is discarded when the tool returns.

Compare this with the two ways we spawned work before.

- In **W07a.2**, the plan node always wrote three sub-questions, and the graph's edges decided that all workers run once, together, right after the plan.
- In **W08a**, a handoff tool returned `Command(goto=..., graph=Command.PARENT)`, and control left the agent for good.

Here the lead is a model in its own loop. It decides how many subagents to create and writes each one's instructions. It reads their reports, and if a report has a gap, it can call `task` again. The `Command` that `task` returns only updates the state, so control stays with the lead.

When the lead calls `task` several times in one message, `ToolNode` runs the calls at the same time, so the subagents work in parallel.

## 7. The lead

The lead gets the to-do list, the file tools, and `task`. It does not get the paper tools. The only way it can learn anything about the literature is to delegate, which keeps the search results out of its context.

In [ ]:
LEAD_PROMPT = """You are the lead of a research team. You cannot search the literature yourself;
subagents do that through the task tool.
1. Start by writing a to-do list with write_todos.
2. Delegate each part of the question with task. Write each description in full, because the subagent
   sees nothing else, and give each one its own notes file under /notes/.
3. Read the notes files you need with read_file. If a part is missing, delegate again.
4. Write the report to /report.md with write_file: what is known, organized by the parts of the question,
   every claim followed by the paper id in brackets as the notes give it.
Call write_todos again whenever an item starts or finishes, so its status is always current.
When /report.md is written, call write_todos to mark the last item completed, then reply with one sentence saying so."""

lead = build_loop(model, LEAD_PROMPT, [write_todos, task, read_file, list_files, write_file])

In [ ]:
start = time.time()

run = lead.invoke({"messages": [HumanMessage(content=QUESTION)]},
                  {"recursion_limit": 60})

seconds = time.time() - start


In [ ]:
run["messages"]

In [ ]:
for name, arg in team.tool_calls_of(run["messages"]):
    print(f"  {name:12} {arg}")
print(f"\n{seconds:.0f} s")

### The instructions the lead wrote

Each description below is the entire context of one subagent. If a description leaves something out, such as which part of the question it covers or where to save its notes, the subagent has no way to find out.

In [ ]:
for m in run["messages"]:
    for call in getattr(m, "tool_calls", None) or []:
        if call["name"] == "task":
            print(call["args"]["description"], "\n")

### The to-do list, the files, and the report

In [ ]:
for t in run["todos"]:
    print(f"[{t['status']:11}] {t['content']}")
print()
for path, text in run["files"].items():
    print(f"{path:45} {len(text):6,} characters")

In [ ]:
from IPython.display import Markdown, display

report = Markdown(run["files"].get("/report.md", "No report was written."))
display(report)

Look at the statuses in the to-do list. The prompt asks the lead to update the list when an item starts or finishes.
- In some runs the lead marks the three research items completed after the subagents report back.
- In others it never calls `write_todos` a second time. A lead that delegates all three parts in one message has no step between starting them and finishing them.

The list is more useful on tasks the agent works through one step at a time. We'll see the library's version of this tool in the next notebook.

## 8. Comparing the two runs

Let's put the two runs side by side. For the harness, we count the lead's tokens and the subagents' tokens separately.

In [ ]:
report = run["files"].get("/report.md", "")
lead_tokens = team.tokens_of(run["messages"])

print(f"{'':28}{'single agent':>14}{'harness':>14}")
print(f"{'tool calls (lead)':28}{len(one['calls']):>14}{len(team.tool_calls_of(run['messages'])):>14}")
print(f"{'largest context (tokens)':28}{max(input_tokens(one['messages'])):>14,}{max(input_tokens(run['messages'])):>14,}")
print(f"{'tokens, lead':28}{one['tokens']:>14,}{lead_tokens:>14,}")
print(f"{'tokens, subagents':28}{0:>14,}{run['subagent_tokens']:>14,}")
print(f"{'tokens, total':28}{one['tokens']:>14,}{lead_tokens + run['subagent_tokens']:>14,}")
print(f"{'papers cited':28}{len(rt.found_ids(one['report'])):>14}{len(rt.found_ids(report)):>14}")
print(f"{'coverage of the gold list':28}{rt.score_coverage(one['report'], GOLD):>14.2f}{rt.score_coverage(report, GOLD):>14.2f}")
print(f"{'seconds':28}{one['seconds']:>14.0f}{seconds:>14.0f}")

A few things to note:

- **Largest context** is the most the model had to read in one call. The single agent's context is small only because it stopped early. Had it read three or four papers for each question, every abstract would still be in its message list. The lead's largest context holds the three notes files it read and the subagents' summaries, and none of the searches or abstracts the subagents read.
- **Total tokens** is the cost. The harness costs more in total, because every subagent reads its own searches and abstracts. Anthropic reported the same pattern for their research system: the multi-agent version used several times the tokens of a single agent.
- **Coverage** is noisy with a small model and a strict gold list, as we saw in Week 7. Run both systems a few times before you conclude that one covers more.

Next, we'll see how the `deepagents` library packages the same pieces.

## 9. Things to try

1. Give the lead `rt.search_papers` and `rt.get_paper` as well. Does it still delegate, or does it start searching itself? Compare the largest context with the run above.

2. Remove "Write each description in full, because the subagent sees nothing else" from `LEAD_PROMPT` and run the lead again. Read the descriptions it writes. What do the subagents get wrong?

3. Change `task` so that it returns the subagent's whole transcript instead of its last message, for example by joining the text of every message in `result["messages"]`. What happens to the lead's largest context?

4. Change `RESEARCHER_PROMPT` so that the subagent does not save notes and instead puts everything in its reply. Does the report change? Does the lead's context?

In [ ]:
# your experiments here